In [ ]:
from datasets import load_dataset

Load Artbench Dataset

In [ ]:
import pandas as pd

df = pd.read_csv('/home/yourname/.cache/kagglehub/datasets/alexanderliao/artbench10/versions/2/ArtBench-10.csv')
#df = pd.read_csv('../../../../codes/artbench/ArtBench-10.csv')
df.head()

In [ ]:
print(df.iloc[0][0])
print(df.iloc[0][1])
print(df.iloc[0][2])

print(df.iloc[1][0])
print(df.iloc[1][1])
print(df.iloc[1][2])

print(df.iloc[2][0])
print(df.iloc[2][1])
print(df.iloc[2][2])

print(df.iloc[3][0])
print(df.iloc[3][1])
print(df.iloc[3][2])

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
from tqdm.notebook import tqdm 

def scrape_wikiart_artist(artist_slug):
    if not isinstance(artist_slug, str):
        return {"artist_slug": artist_slug, "status": "Invalid Name"}
        
    url = f"https://www.wikiart.org/en/{artist_slug}"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return {"artist_slug": artist_slug, "status": f"HTTP {response.status_code}"}
        
        soup = BeautifulSoup(response.text, 'html.parser')
        metadata = {"artist_slug": artist_slug, "status": "Success"}
        info_list = soup.find('ul', class_='p-info-list')
        items = info_list.find_all('li') if info_list else soup.find_all('li')
        
        for li in items:
            text = li.get_text(separator=" ").strip()
            if ":" in text and len(text) < 200:
                parts = text.split(":", 1)
                key = parts[0].strip().lower()
                val = parts[1].strip()
                metadata[key] = val
        
        return metadata
    except Exception as e:
        return {"artist_slug": artist_slug, "status": f"Error: {str(e)}"}

In [ ]:
from tqdm import tqdm 
unique_artists = df['artist'].unique()
print(f"Total unique artists found: {len(unique_artists)}")

test_batch = unique_artists[1000:] 

results = []
for artist in tqdm(test_batch, desc="Scraping WikiArt"):
    data = scrape_wikiart_artist(artist)
    results.append(data)
    time.sleep(1.1) 
wiki_metadata_df2 = pd.DataFrame(results)

print("\n--- Scraped Metadata ---")
display(wiki_metadata_df1.head())

In [ ]:
# 4. Display the results
print("\n--- Scraped Metadata ---")
display(wiki_metadata_df2.head())

In [ ]:
frames = [wiki_metadata_df, wiki_metadata_df1]

result = pd.concat(frames)

In [ ]:
display(result.head())

In [ ]:
import json

wiki_metadata_df2.to_json('wikiart_artist_metadata1.json', orient='records', indent=4)

print("Files saved successfully:")
print("- wikiart_artist_metadata1.json")

In [ ]:
import wikipediaapi
import json
import os

wiki_wiki = wikipediaapi.Wikipedia(
    user_agent='ArtInfluenceResearch/1.0 (contact: yourname@example.com)',
    language='en',
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

def fetch_and_save_artist_wiki(page_title, save_dir="./"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    page = wiki_wiki.page(page_title)
    
    if not page.exists():
        return None
    
    def get_sections(sections):
        return {s.title: {"text": s.text, "subsections": get_sections(s.sections)} for s in sections}

    artist_data = {
        "title": page.title,
        "summary": page.summary,
        "full_text": page.text,
        "url": page.fullurl,
        "categories": [c.replace("Category:", "") for c in page.categories.keys()],
        "sections": get_sections(page.sections)
    }
    
    file_path = os.path.join(save_dir, f"{page_title.replace(' ', '_')}.json")
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(artist_data, f, ensure_ascii=False, indent=4)
    
    print(f"Wikipedia Data saved: {file_path}")
    return artist_data

In [ ]:
import json
import time
from tqdm import tqdm

with open('wikiart_artist_metadata1.json', 'r', encoding='utf-8') as f:
    artists_list = json.load(f)

print(f"Loaded {len(artists_list)} artists from JSON.")

def extract_wiki_title(wiki_field):
    """Extracts 'Frank O'Meara' from 'en.wikipedia.org/wiki/Frank_O'Meara'"""
    if not wiki_field or not isinstance(wiki_field, str):
        return None
    if "/wiki/" in wiki_field:
        title = wiki_field.split("/wiki/")[-1]
        return title.replace('_', ' ')
    return None

all_wiki_data = []

for artist in tqdm(artists_list, desc="Fetching Wikipedia Data"):
    page_title = extract_wiki_title(artist.get("wikipedia"))
    
    if not page_title:
        slug = artist.get("artist_slug", "")
        page_title = " ".join(word.capitalize() for word in slug.split("-")[::-1])

    try:
        wiki_data = fetch_and_save_artist_wiki(page_title, save_dir="./artist_wiki_pages")
        if wiki_data:
            all_wiki_data.append(wiki_data)
        
        time.sleep(0.5) 
    except Exception as e:
        print(f"Error processing {page_title}: {e}")

print(f"\nSuccessfully fetched {len(all_wiki_data)} Wikipedia articles.")

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import json

sparql = SPARQLWrapper("http://dbpedia.org/sparql")

In [ ]:
def fetch_artist_full_knowledge(artist_name):
    resource_slug = artist_name.replace(" ", "_")
    resource_url = f"http://dbpedia.org/resource/{resource_slug}"
    
    sparql = SPARQLWrapper("http://dbpedia.org/sparql")
    
    query = f"""
    PREFIX dbo: <http://dbpedia.org/ontology/>
    PREFIX dbp: <http://dbpedia.org/property/>
    PREFIX dct: <http://purl.org/dc/terms/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?rel ?val WHERE {{
      {{ <{resource_url}> dbo:artMovement ?val . BIND("movement" AS ?rel) }}
      UNION
      {{ <{resource_url}> dbp:movement ?val . BIND("movement" AS ?rel) }}
      UNION
      {{ <{resource_url}> dbo:influencedBy ?val . BIND("influencedBy" AS ?rel) }}
      UNION
      {{ <{resource_url}> dbp:knownFor ?val . BIND("knownFor" AS ?rel) }}
      UNION
      {{ <{resource_url}> dct:subject ?val . BIND("category" AS ?rel) }}
      UNION
      {{ <{resource_url}> dbo:wikiPageWikiLink ?val . BIND("connection" AS ?rel) }}
    }}
    """
    
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    
    try:
        results = sparql.query().convert()
        data = {"movement": [], "influencedBy": [], "categories": [], "connections": []}
        
        for res in results["results"]["bindings"]:
            rel = res["rel"]["value"]
            val = res["val"]["value"].split('/')[-1].replace('_', ' ')
            
            if rel == "movement" or rel == "knownFor": data["movement"].append(val)
            elif rel == "influencedBy": data["influencedBy"].append(val)
            elif rel == "category": data["categories"].append(val)
            elif rel == "connection": data["connections"].append(val)
            
        for k in data: data[k] = list(set(data[k]))
        return data
    except Exception as e:
        return None

frank_knowledge = fetch_artist_full_knowledge("Frank_O'Meara")
print(json.dumps(frank_knowledge, indent=2))

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import json

def fetch_artist_triples(artist_name):
    resource_slug = artist_name.replace(" ", "_")
    resource_url = f"http://dbpedia.org/resource/{resource_slug}"
    
    sparql = SPARQLWrapper("http://dbpedia.org/sparql")
    
    # We select EVERYTHING (?p = predicate, ?o = object)
    # We use <{resource_url}> to handle apostrophes safely
    query = f"""
    SELECT ?p ?o WHERE {{
      <{resource_url}> ?p ?o .
    }}
    """
    
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    
    try:
        results = sparql.query().convert()
        triples = []
        
        for res in results["results"]["bindings"]:
            predicate = res["p"]["value"]
            obj = res["o"]["value"]
            
            triples.append({
                "subject": resource_url,
                "predicate": predicate,
                "object": obj
            })
            
        return triples
    except Exception as e:
        print(f"Error for {artist_name}: {e}")
        return []

# Test for Frank O'Meara
frank_triples = fetch_artist_triples("Frank_O'Meara")

print(f"Total Triples Found: {len(frank_triples)}")
print(json.dumps(frank_triples[:5], indent=2))

In [ ]:
import pandas as pd
import json
import os

def save_artist_kg(artist_name, triples, folder="./"):
    if not os.path.exists(folder):
        os.makedirs(folder)
    
    kg_df = pd.DataFrame(triples)
    
    raw_path = os.path.join(folder, f"{artist_name}_raw_kg.csv")
    kg_df.to_csv(raw_path, index=False)
    
    clean_df = kg_df.copy()
    clean_df['predicate'] = clean_df['predicate'].apply(lambda x: x.split('/')[-1].split('#')[-1])
    clean_df['object'] = clean_df['object'].apply(lambda x: x.split('/')[-1] if 'http' in str(x) else x)
    
    clean_path = os.path.join(folder, f"{artist_name}_clean_kg.csv")
    clean_df.to_csv(clean_path, index=False)
    
    print(f"Graph nodes saved for {artist_name}:")
    print(f" - Raw: {raw_path}")
    print(f" - Clean: {clean_path}")
    return clean_df

frank_triples = fetch_artist_triples("Frank_O'Meara")

if frank_triples:
    clean_kg = save_artist_kg("Frank_O'Meara", frank_triples)
    
    print("\n--- Knowledge Graph Preview (Cleaned) ---")
    display(clean_kg.head(10))

new dbpedia

In [ ]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm
from SPARQLWrapper import SPARQLWrapper, JSON
from urllib.parse import unquote

sparql = SPARQLWrapper("http://dbpedia.org/sparql")
sparql.setTimeout(30) 
OUTPUT_FOLDER = "./artist_knowledge_graphs"


def get_dbpedia_resource_name(wiki_field):
    """Extracts the identifier from the Wikipedia URL."""
    if not wiki_field or not isinstance(wiki_field, str):
        return None
    if "/wiki/" in wiki_field:
        resource_part = wiki_field.split("/wiki/")[-1]
        return unquote(resource_part)
    return None

def fetch_artist_triples(resource_name):
    """Queries DBpedia for all triples associated with the artist."""
    query = f"""
    SELECT ?predicate ?object
    WHERE {{
      <http://dbpedia.org/resource/{resource_name}> ?predicate ?object .
    }}
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    
    try:
        results = sparql.query().convert()
        bindings = results["results"]["bindings"]
        return [{"subject": f"http://dbpedia.org/resource/{resource_name}", 
                 "predicate": b["predicate"]["value"], 
                 "object": b["object"]["value"]} for b in bindings]
    except Exception as e:
        return None

def save_artist_kg(artist_name, triples, folder):
    """Saves both raw and cleaned versions of the knowledge graph."""
    if not os.path.exists(folder):
        os.makedirs(folder)
    
    safe_name = artist_name.replace("'", "").replace('"', "").replace("/", "_").replace(":", "_")
    
    df = pd.DataFrame(triples)
    
    raw_path = os.path.join(folder, f"{safe_name}_raw.csv")
    df.to_csv(raw_path, index=False)
    
    clean_df = df.copy()
    clean_df['predicate'] = clean_df['predicate'].apply(lambda x: x.split('/')[-1].split('#')[-1])
    clean_df['object'] = clean_df['object'].apply(lambda x: x.split('/')[-1] if 'http' in str(x) else x)
    clean_df['subject'] = artist_name
    
    clean_path = os.path.join(folder, f"{safe_name}_clean.csv")
    clean_df.to_csv(clean_path, index=False)


with open('wikiart_artist_metadata1.json', 'r', encoding='utf-8') as f:
    artists_list = json.load(f)

print(f"Total artists to process: {len(artists_list)}")

for artist in tqdm(artists_list, desc="Scraping DBpedia"):
    wiki_link = artist.get("wikipedia")
    res_name = get_dbpedia_resource_name(wiki_link)
    
    if res_name:
        safe_name = res_name.replace("'", "").replace('"', "").replace("/", "_").replace(":", "_")
        if os.path.exists(os.path.join(OUTPUT_FOLDER, f"{safe_name}_raw.csv")):
            continue 

        data = fetch_artist_triples(res_name)
        
        if data:
            save_artist_kg(res_name, data, OUTPUT_FOLDER)
        else:

            time.sleep(2) 
    
    time.sleep(0.4)

print(f"\nData saved in: {OUTPUT_FOLDER}")